# triage-lens: transformer models on a Kaggle GPU

Thin runner: all logic lives in the repository (`src/triage_lens/transformer.py`).
The notebook clones it, fine-tunes DistilBERT and runs a zero-shot NLI model, and
writes class probabilities for the validation and test splits. They are scored
locally with `python -m triage_lens.predictions`, the same way as every other model.

**Settings:** Accelerator → GPU (T4 or P100), Internet → on.
**Input:** a private Kaggle dataset with `astral-sh__uv.dataset.jsonl` and
`pandas-dev__pandas.dataset.jsonl` from `data/processed/`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone --depth 1 https://github.com/IlyaShaposhnikov/triage-lens.git
# Kaggle already has torch, transformers, numpy and scikit-learn: keep its versions.
!pip install -q -e ./triage-lens --no-deps
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| cuda", torch.cuda.is_available())

In [ ]:
import glob, os
matches = glob.glob("/kaggle/input/**/astral-sh__uv.dataset.jsonl", recursive=True)
assert matches, "Attach the dataset with the *.dataset.jsonl files (Add Input)"
DATA = os.path.dirname(matches[0])
print(DATA, sorted(os.listdir(DATA)))

## 1. Smoke run (about a minute)

200 issues, one epoch: checks the whole pipeline before the long runs.
Output goes to a separate folder and is not used.

In [ ]:
!cd triage-lens && python -m triage_lens.transformer finetune astral-sh/uv --data {DATA} --out /tmp/smoke --work-dir /tmp/checkpoints --limit 200 --epochs 1
!cd triage-lens && python -m triage_lens.transformer zeroshot astral-sh/uv --data {DATA} --out /tmp/smoke --limit 40

## 2. Fine-tune DistilBERT (about 10–25 minutes for both repositories)

In [ ]:
!cd triage-lens && python -m triage_lens.transformer finetune astral-sh/uv pandas-dev/pandas --data {DATA} --out /kaggle/working/predictions --work-dir /tmp/checkpoints

## 3. Zero-shot baseline (bart-large-mnli, about 15–30 minutes)

In [ ]:
!cd triage-lens && python -m triage_lens.transformer zeroshot astral-sh/uv pandas-dev/pandas --data {DATA} --out /kaggle/working/predictions

## 4. Download

Download `predictions.zip` from the Output panel and unpack it into the repository
root, so the files land in `predictions/<owner__name>/`.

In [ ]:
!cd /kaggle/working && zip -qr predictions.zip predictions && ls -la predictions/*